In [2]:
!pip install transformers datasets

In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, pipeline
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from datasets import Dataset, load_dataset
import json
import torch
import torch.nn as nn
import torch.nn.functional as F

import pandas as pd

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
# Load the food.csv file into a DataFrame
df_food = pd.read_csv('/content/drive/MyDrive/food.csv')
df_ingredients = pd.read_csv('/content/drive/MyDrive/branded_food.csv')


# select needed columns, ignoring every other column
selected_columns_food= df_food[['fdc_id']]
selected_columns_ingredients = df_ingredients[['fdc_id', 'ingredients']]




/tmp/ipython-input-1834772481.py:2: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df_food = pd.read_csv('/content/drive/MyDrive/food.csv')
/tmp/ipython-input-1834772481.py:3: DtypeWarning: Columns (2,3,4,6,12,16,17,18,19) have mixed types. Specify dtype option on import or set low_memory=False.
  df_ingredients = pd.read_csv('/content/drive/MyDrive/branded_food.csv')


In [6]:
# merge two dataframes on fdc_id

merged_df = pd.merge(selected_columns_food, selected_columns_ingredients, on='fdc_id', how='inner')
print(merged_df.shape)

(1977398, 2)


In [7]:
#Dataset from merged dataframe
dataset = Dataset.from_pandas(merged_df.head(10000))

print(dataset.shape)

allergens = []

(10000, 2)


In [8]:
def clean_data(example):
    ingredients = example['ingredients']
    # Check if ingredients is None or if it's an empty string after stripping whitespace
    if ingredients is None or (isinstance(ingredients, str) and ingredients.strip() == ''):
        return False
    return True

# Apply filter to remove invalid rows
dataset = dataset.filter(clean_data)

print(dataset.features)
print("Dataset filtered successfully.")
print(dataset.shape)

Filter:   0%|          | 0/10000 [00:00<?, ? examples/s]

{'fdc_id': Value('int64'), 'ingredients': Value('string')}
Dataset filtered successfully.
(10000, 2)


In [9]:
allergens = [
    "butter", "casein",
    "cheese", "cream", "curd", "custard", "ghee", "half-and-half",
    "lactalbumin", "lactalbumin phosphate", "lactic acid starter culture", "lactoferrin",
    "lactoglobulin", "lactose", "lactulose", "milk",
    "pudding", "recaldent", "simplesse", "tagatose", "whey",
    "yogurt", "albumin","albumen", "apovitellin",
    "avidin globulin", "egg", "eggnog", "lysozyme",
    "mayonnaise", "meringue", "ovalbumin", "ovomucoid", "ovomucin",
    "ovovitellin", "surimi", "vitellin", "arachis oil",
    "lupin", "mandelonas", "soy",  "edamame", "miso", "natto", "okara", "shoyu",
    "tamari", "tempeh",
    "textured vegetable protein", "tvp", "tofu", "bread crumbs", "bulgur",
    "cereal extract", "couscous", "cracker meal", "einkorn", "emmer", "matzo",
    "matza", "pasta", "seitan", "semolina", "peanut",
    "spelt", "triticale",
    "wheat","bran", "durum", "germ", "gluten", "grass", "malt", "sprouts", "starch","almond",
    "cashew", "filbert", "gianduja", "marzipan",
    "nut", "pecan", "pesto",
    "pistachio", "praline", "barnacle", "crab", "crawfish", "crayfish", "ecrevisse",
    "krill", "lobster", "langouste", "langoustine", "moreton bay bugs", "scampi", "tomalley",
    "prawns", "shrimp", "crevette", "benne", "benniseed", "gingelly", "gomasio", "halvah", "sesame", "sesamol", "sesamum indicum", "sim sim",
    "tahini", "tahina", "tehina", "til",    "anchovies", "bass","catfish","cod", "flounder", "grouper",
    "haddock", "hake", "halibut", "herring", "mahi mahi","perch","pike","pollock",
    "salmon","scrod","sole","snapper","swordfish","tilapia","trout","tuna","surimi"
    ]

allergens = {
    "milk": [
        "butter", "casein", "cheese", "cream", "curd", "custard", "ghee", "half-and-half",
        "lactalbumin", "lactalbumin phosphate", "lactic acid starter culture", "lactoferrin",
        "lactoglobulin", "lactose", "lactulose", "milk", "pudding", "recaldent", "simplesse",
        "tagatose", "whey", "yogurt"
    ],
    "eggs": [
        "albumin", "albumen", "apovitellin", "avidin globulin", "egg", "eggnog", "lysozyme",
        "mayonnaise", "meringue", "ovalbumin", "ovomucoid", "ovomucin", "ovovitellin",
        "surimi", "vitellin"
    ],
    "fish": [
        "anchovies", "bass", "catfish", "cod", "flounder", "grouper", "haddock", "hake",
        "halibut", "herring", "mahi mahi", "perch", "pike", "pollock", "salmon", "scrod",
        "sole", "snapper", "swordfish", "tilapia", "trout", "tuna", "surimi"
    ],
    "shellfish": [
        "barnacle", "crab", "crawfish", "crayfish", "ecrevisse", "krill", "lobster",
        "langouste", "langoustine", "moreton bay bugs", "scampi", "tomalley", "prawns",
        "shrimp", "crevette"
    ],
    "nuts": [
        "arachis oil", "lupin", "mandelonas", "peanut", "almond", "cashew", "filbert",
        "gianduja", "marzipan", "nut", "pecan", "pesto", "pistachio", "praline"
    ],
    "soy": [
        "soy", "edamame", "miso", "natto", "okara", "shoyu", "tamari", "tempeh",
        "textured vegetable protein", "tvp", "tofu"
    ],
    "wheat": [
        "bread crumbs", "bulgur", "cereal extract", "couscous", "cracker meal", "einkorn",
        "emmer", "matzo", "matza", "pasta", "seitan", "semolina", "spelt", "triticale",
        "wheat", "bran", "durum", "germ", "gluten", "grass", "malt", "sprouts", "starch"
    ],
    "sesame": [
        "benne", "benniseed", "gingelly", "gomasio", "halvah", "sesame", "sesamol",
        "sesamum indicum", "sim sim", "tahini", "tahina", "tehina", "til"
    ]
}

def detect_allergens(dataset):
    ingredients = str(dataset['ingredients']).lower()

    for category, keywords in allergens.items():
        dataset[category + "_present"] = int(
            any(keyword in ingredients for keyword in keywords)
        )

    return dataset

def assign_label(dataset):
    dataset["label"] = int(
        dataset["milk_present"] +
        dataset["eggs_present"] +
        dataset["fish_present"] +
        dataset["shellfish_present"] +
        dataset["nuts_present"] +
        dataset["soy_present"] +
        dataset["wheat_present"] +
        dataset["sesame_present"] > 0
    )
    return dataset
# Apply the function to the dataset
dataset_with_allergens = dataset.map(detect_allergens)
dataset_with_allergens = dataset_with_allergens.map(assign_label)

print(dataset_with_allergens.features)
print(dataset_with_allergens[0])


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

{'fdc_id': Value('int64'), 'ingredients': Value('string'), 'milk_present': Value('int64'), 'eggs_present': Value('int64'), 'fish_present': Value('int64'), 'shellfish_present': Value('int64'), 'nuts_present': Value('int64'), 'soy_present': Value('int64'), 'wheat_present': Value('int64'), 'sesame_present': Value('int64'), 'label': Value('int64')}
{'fdc_id': 1105904, 'ingredients': 'Vegetable Oil', 'milk_present': 0, 'eggs_present': 0, 'fish_present': 0, 'shellfish_present': 0, 'nuts_present': 0, 'soy_present': 0, 'wheat_present': 0, 'sesame_present': 0, 'label': 0}


In [10]:
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    texts = [str(text) for text in batch["ingredients"]]

    tok = tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=256
    )

    tok["labels"] = batch["label"]
    return tok

# Tokenize the dataset
tokenized_ds = dataset_with_allergens.map(tokenize, batched=True)

print(tokenized_ds.features)

# Split the dataset into train and test sets
tokenized_ds = tokenized_ds.train_test_split(test_size=0.15)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

{'fdc_id': Value('int64'), 'ingredients': Value('string'), 'milk_present': Value('int64'), 'eggs_present': Value('int64'), 'fish_present': Value('int64'), 'shellfish_present': Value('int64'), 'nuts_present': Value('int64'), 'soy_present': Value('int64'), 'wheat_present': Value('int64'), 'sesame_present': Value('int64'), 'label': Value('int64'), 'input_ids': List(Value('int32')), 'token_type_ids': List(Value('int8')), 'attention_mask': List(Value('int8')), 'labels': Value('int64')}


In [11]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary'
    )
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

In [ ]:
num_labels = 2

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

training_args = TrainingArguments(
    eval_strategy="epoch",
    num_train_epochs=3,
    per_device_train_batch_size=10,
    per_device_eval_batch_size=10,
    learning_rate=2e-5,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"],
    compute_metrics=compute_metrics
)

trainer.train()

clf = pipeline(
    "text-classification",
    model=model,
    tokenizer=model_name
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Find your API key here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: yunjj (yunjj-occidental-college) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.175600,0.022904,0.995333,0.999039,0.994264,0.996646
2,0.024600,0.026485,0.994000,0.998079,0.993308,0.995688
3,0.010100,0.024410,0.995333,0.996180,0.997132,0.996656


Device set to use cuda:0


In [15]:
text = "Ingredients: sugar"
result = clf(text)
print(result)

[{'label': 'LABEL_0', 'score': 0.999803364276886}]
